In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Annotated
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.messages import HumanMessage
import sqlite3
from dotenv import load_dotenv
from pathlib import Path
from langgraph.checkpoint.sqlite import SqliteSaver

In [2]:
load_dotenv()

True

In [3]:
drive_folder = Path(r"G:\My Drive\SynapseAI")
drive_folder.mkdir(parents=True, exist_ok=True)

db_path = drive_folder / "synapse.db"

conn = sqlite3.connect(
    str(db_path),
    check_same_thread=False
)

checkpointer = SqliteSaver(conn)
checkpointer.setup()

print("Database path:", db_path)

Database path: G:\My Drive\SynapseAI\synapse.db


In [4]:
nv=ChatNVIDIA(
    model='nvidia/nemotron-3.5-super-vl-120b-a12b'
)

c:\Users\pushk\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:250: UserWarning: Found nvidia/nemotron-3.5-super-vl-120b-a12b in available_models, but type is unknown and inference may fail.
  warnings.warn(


In [5]:
diffgemma=ChatNVIDIA(
    model='google/diffusiongemma-26b-a4b-it'
)

c:\Users\pushk\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:250: UserWarning: Found google/diffusiongemma-26b-a4b-it in available_models, but type is unknown and inference may fail.
  warnings.warn(


In [6]:
gemma=ChatNVIDIA(
    model='google/gemma-4-31b-it'
)

In [7]:
model=ChatGroq(
    model="openai/gpt-oss-120b"
)

In [8]:
class ChatbotState(TypedDict):
    chat_history: Annotated[list,add_messages]

In [9]:
def chat(state: ChatbotState):
    ans = model.invoke(state['chat_history'])
    return {'chat_history': [ans]}

In [10]:
graph = StateGraph(ChatbotState)

graph.add_node('chat', chat)

graph.add_edge(START, 'chat')
graph.add_edge('chat', END)

workflow = graph.compile(checkpointer=checkpointer)

In [11]:
config={'configurable':{"thread_id":"chat-001"}}

while True:
  user_input = input('\nYou: ').strip()
  if user_input.lower() in ['exit', 'bye', 'quit']:
      print('\nChatbot: Goodbye!')
      break

  if not user_input:
      continue

  for message_chunk,metadata in workflow.stream(
      {'chat_history':[HumanMessage(content=user_input)]},
      config=config,
      stream_mode='messages'
  ):
    if message_chunk.content:
      print(message_chunk.content,end="",flush=True)

Hello! How can I help you today?
Chatbot: Goodbye!
